# Transfusion 一个模型里做自回归文本和图像扩散

Chameleon和Emu3在离散token上下押。这有用，但是量化瓶颈可见————图像质量低于连续空间的扩散模型。Transfusion押了对立的一手，图像保持连续，完全丢掉VQ-VAE，然后训练一个有两种损失的transformer。对于文本，做下一词元预测。对于图像patch，做流匹配/扩散损失。这两个目标都在同一套权重下做优化。架构与Stable Diffusion3 （MMDiT）是近亲。

## 问题描述

离散还是连续的图像token之争比LLM还要早。连续的表征（原始像素、VAE潜空间）能保住细节。离散token（VQ 索引）适合transformer的原生词表，不过在量化步骤中丢掉了细节。

Chameleon和Emu3 走的是离散线路：一个损失，一个架构，不过图像保真度被分词器质量限制。

扩散模型走的是连续路线：图像质量出奇的好，但是独立于LLM的单独模型，有复杂的噪声调度，与文本生成集成困难。

Transfusion 问：能把两者弄到一起吗？保留图像的连续性，但还是只训练一个模型，在一个梯度步中将两种损失缝到一起。

## 基本概念

### 两种损失架构

一个仅解码的Transformer处理包含以下内容的序列：
- 文本tokens。 离散，从BPE词表获取
- 图像patches。 连续，16x16的像素块通过线性嵌入投影到隐空间，就像ViT的编码器输入一样。
- `<image>`和`</image>`标记，标注连续块出现的位置。

前向只跑一次，每个token的损失从两个头中选一个：
- 对于文本词元：词表logits头上的标准交叉熵。
- 对于图片patches：连续patches上的扩散损失，预测加给每个patch的噪声。

梯度流过共享的transformer主体，两个损失同时改进共享权重。

### 注意力掩码：因果文本+双向图像

文本token必须是因果的，你不能让一个文本token偷看未来文本，否则teacher forcing就坏了。对于图像patches来说，一个图像块内它们应该彼此双向关注。

掩码：
```
N[i, j] = 1 if:
(i is text and j is text and j <= i)  # 文本因果
OR (i is image and j is image and same_image_block(i, j)) # 图像块内双向关注
OR (i is text and j is image and j < i_image_end) # 文本可以看到之前的图像
OR (i is image and j is text and j < i_image_start) # 图像能看到以前的文本
```

### Transformer中的扩散损失

标准的扩散损失是：对图像块加噪，然后让模型预测加的噪声（或者干净的图形，等价的）。Transfusion 用的是流动匹配，预测从噪音到干净图片的速度场。

训练过程中：
- 对每个图像patch x0，采样随机事件步t。
- 对噪声e采样，然后计算出xt = (1-t)*x0 + t*e，流匹配线性插值。
- 然后transformer预测v_theta(xt, t);损失是MSE
- 相同序列沿着文本NTP损失反向传播。

推理过程中，生成是：
- 文本token： 标准的自回归采样。
- 图片patches： 基于前缀文本token的扩散采样循环（10-30步）

### MMDiT：Stable Diffusion 3 的变体

Stable Diffusion 3 在Transfusion 的同时间内发表了 MMDiT（Multimodal Diffusion Transformer）。 这两个架构是兄弟：
MMDiT的关键不同：
- 每个块有模块专门的权重。每个transformer块针对文本token和图片patches有专门的Q、K、V以及MLP权重。注意力是联合的（跨模态）。其余一切都是模态专属的。
- Rectified flow training。 一个特定的flow-matching变体，采样已知且数学比DDPM简单。
- 规模。MMDiT是SD3的骨架（2B和8B两个变体），Transfusion 的参数量为7B。

二者都收敛到相同的核心思想：同一个transformer在文本上跑下一token预测，在连续的图片表示上跑扩散。

### 为什么能击败Chameleon

连续扩散和离散NTP在图像生成上的差距是可测量的，Transfusion的论文表明：
- 7B的参数，在FID上击败了同尺寸的Chameleon 3-5分
- 不需要训练分词器，图像编码器更简单（线性投影到隐空间，跟ViT的输入层一样）
- 推理可以并行图片去噪，不像自回归的图像token。

缺点：Transfusion是一个双损失模型，训练动态棘手。损失权重需要微调，NTP和扩散之间的调度不匹配会让其中某个头变成主导。

### 下游

Janus-Pros 通过将用于理解和生成的视觉编码器接耦，一个用SigLIP，另一个用VQ，同时共享Transformer主体。Show-o把扩散替换成离散扩散（掩码预测）。统一生成家族在Transfusion之后迅速分叉。

2026年那些能吐图像的生产VLM，生成路径，几乎肯定用了这个家族的某个后代。细节是专有的。

# 开始编码

教学积木：Transfusion 核心——**连续 patch 嵌入**、**因果文本 + 双向图像注意力掩码**、**流匹配双损失**、**预留槽位并行去噪推理**。玩具规模，不追求复现论文数字。


## 1. 配置、连续 patch 嵌入与流匹配工具


In [ ]:
from __future__ import annotations

from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class TinyTransfusionConfig:
    """Transfusion 教学配置（远小于真模型）。"""

    image_size: int = 32
    """输入方图边长。"""

    patch_size: int = 8
    """每个空间 patch 的边长（真文常用 16）。"""

    dim: int = 64
    """Transformer 隐维。"""

    n_heads: int = 4
    n_layers: int = 2
    dropout: float = 0.0
    """教学默认关掉 dropout，便于冒烟复现。"""

    text_vocab_size: int = 128
    """文本词表大小（含特殊符号）。"""

    id_bos: int = 1
    id_eos: int = 2
    id_img_start: int = 3
    id_img_end: int = 4
    """`<image>` / `</image>` 在文本词表中的 id。"""

    text_loss_weight: float = 1.0
    image_loss_weight: float = 1.0
    """双损失加权；真训练里常需要仔细调度。"""

    @property
    def grid_side(self) -> int:
        """空间网格边长（patch 行/列数）。"""
        if self.image_size % self.patch_size:
            raise ValueError("image_size must be divisible by patch_size")
        return self.image_size // self.patch_size

    @property
    def num_patches(self) -> int:
        """一张图展开后的连续 patch 个数。"""
        return self.grid_side * self.grid_side

    @property
    def patch_dim(self) -> int:
        """单个 patch 展平后的原始维（C*P*P，这里 C=3）。"""
        return 3 * self.patch_size * self.patch_size


class PatchEmbed(nn.Module):
    """把 RGB 图切成连续 patch，线性投到隐空间（类 ViT，无 VQ）。"""

    def __init__(self, cfg: TinyTransfusionConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.proj = nn.Linear(cfg.patch_dim, cfg.dim)
        self.unproj = nn.Linear(cfg.dim, cfg.patch_dim)

    def encode(self, images: torch.Tensor) -> torch.Tensor:
        """
        Args:
            images: ``(B, 3, H, W)``，``H=W=image_size``。

        Returns:
            patches: ``(B, N, patch_dim)`` 连续像素块（未嵌入）。
        """
        cfg = self.cfg
        B, C, H, W = images.shape
        if (H, W, C) != (cfg.image_size, cfg.image_size, 3):
            raise ValueError("unexpected image shape")
        p = cfg.patch_size
        g = cfg.grid_side
        x = images.unfold(2, p, p).unfold(3, p, p)  # (B,3,g,g,p,p)
        x = x.permute(0, 2, 3, 1, 4, 5).contiguous()
        return x.view(B, g * g, cfg.patch_dim)

    def embed(self, patches: torch.Tensor) -> torch.Tensor:
        """
        Args:
            patches: ``(B, N, patch_dim)``。

        Returns:
            h: ``(B, N, D)``。
        """
        return self.proj(patches)

    def decode_patches(self, patches: torch.Tensor) -> torch.Tensor:
        """
        Args:
            patches: ``(B, N, patch_dim)``。

        Returns:
            images: ``(B, 3, H, W)``。
        """
        cfg = self.cfg
        B, N, _ = patches.shape
        g, p = cfg.grid_side, cfg.patch_size
        if N != cfg.num_patches:
            raise ValueError("N must equal num_patches")
        x = patches.view(B, g, g, 3, p, p)
        x = x.permute(0, 3, 1, 4, 2, 5).contiguous()
        return x.view(B, 3, cfg.image_size, cfg.image_size)

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        """
        Args:
            images: ``(B, 3, H, W)``。

        Returns:
            h: ``(B, N, D)`` 嵌入后的图像 token。
        """
        return self.embed(self.encode(images))


def sample_flow_matching(
    x0: torch.Tensor,
    t: torch.Tensor,
    noise: torch.Tensor | None = None,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    线性流匹配：``x_t = (1-t) x0 + t ε``，目标速度 ``v = ε - x0``。

    Args:
        x0: ``(B, N, P)`` 干净连续 patch。
        t: ``(B,)`` 或 ``(B, 1, 1)``，取值 ``[0, 1]``。
        noise: 可选 ``ε``，形状与 ``x0`` 相同；``None`` 则标准正态采样。

    Returns:
        xt: 加噪后的 ``(B, N, P)``。
        v_target: ``(B, N, P)`` 速度场目标。
        noise: 实际使用的 ``ε``。
    """
    if noise is None:
        noise = torch.randn_like(x0)
    while t.ndim < x0.ndim:
        t = t.unsqueeze(-1)
    xt = (1.0 - t) * x0 + t * noise
    v_target = noise - x0
    return xt, v_target, noise


def flow_step(xt: torch.Tensor, v: torch.Tensor, dt: float) -> torch.Tensor:
    """
    欧拉积分一步：从噪声端 ``t=1`` 往数据端 ``t=0`` 走时 ``dt<0``。

    Args:
        xt: ``(B, N, P)`` 当前状态。
        v: ``(B, N, P)`` 预测速度（``ε - x0`` 方向）。
        dt: 时间增量（去噪时常为负数）。

    Returns:
        x_next: ``(B, N, P)``。
    """
    return xt + dt * v


print(
    f"config/patch/flow ready | "
    f"grid={TinyTransfusionConfig().grid_side} N={TinyTransfusionConfig().num_patches}"
)


## 2. 注意力掩码：因果文本 + 同块双向图像


In [ ]:
def build_transfusion_attn_mask(
    is_image: torch.Tensor,
    image_block_id: torch.Tensor,
) -> torch.Tensor:
    """
    构造 Transfusion 风格布尔掩码：``True`` 表示 **允许** 注意。

    规则（与笔记对应）：
    - 文本→文本：因果 ``j <= i``
    - 图像→图像：同一 ``image_block_id`` 内全双向
    - 文本→图像：可见 **序列位置更早** 的图像 token（``j < i``）
    - 图像→文本：可见 **图像块开始之前** 的文本（``j < i`` 且 ``j`` 为文本）

    Args:
        is_image: ``(L,)`` bool，该位是否为连续图像 patch（不含边界符）。
        image_block_id: ``(L,)`` long，同属一块图的 id；非图像位填 ``-1``。

    Returns:
        allow: ``(L, L)`` bool 注意力允许掩码。
    """
    L = is_image.numel()
    device = is_image.device
    i = torch.arange(L, device=device).view(L, 1)
    j = torch.arange(L, device=device).view(1, L)

    text_i = ~is_image
    text_j = ~is_image
    img_i = is_image
    img_j = is_image

    causal_tt = text_i & text_j & (j <= i)
    same_block = (
        img_i
        & img_j
        & (image_block_id.view(L, 1) >= 0)
        & (image_block_id.view(L, 1) == image_block_id.view(1, L))
    )
    # 跨模态 / 跨更早图像：仅看过去位置；同块内未来 patch 由 same_block 放开
    look_back = (j < i) & (
        (text_i & img_j) | (img_i & text_j) | (img_i & img_j)
    )
    return causal_tt | same_block | look_back


def modality_flags_from_layout(
    text_prefix_len: int,
    num_patches: int,
    text_suffix_len: int = 0,
    block_id: int = 0,
    device: torch.device | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    按 ``prefix + <image> + N patches + </image> + suffix`` 生成模态标记。

    边界符 ``<image>`` / ``</image>`` 视为 **文本**（离散 token）。

    Args:
        text_prefix_len: 前缀文本长度（不含 start 符）。
        num_patches: 连续 patch 数 ``N``。
        text_suffix_len: 后缀文本长度（不含 end 符已计入布局时传纯后缀）。
        block_id: 本图像块 id。
        device: 可选设备。

    Returns:
        is_image: ``(L,)``。
        image_block_id: ``(L,)``。
    """
    # layout: [prefix][start][patches][end][suffix]
    L = text_prefix_len + 1 + num_patches + 1 + text_suffix_len
    is_image = torch.zeros(L, dtype=torch.bool, device=device)
    image_block_id = torch.full((L,), -1, dtype=torch.long, device=device)
    img_start = text_prefix_len + 1
    img_end = img_start + num_patches
    is_image[img_start:img_end] = True
    image_block_id[img_start:img_end] = block_id
    return is_image, image_block_id


print("attn mask helpers ready")


## 3. 双头 Transformer：文本 CE + 图像速度场


In [ ]:
class MaskedAttention(nn.Module):
    """带显式布尔允许掩码的多头自注意力。"""

    def __init__(self, dim: int, n_heads: int, dropout: float) -> None:
        super().__init__()
        if dim % n_heads:
            raise ValueError("dim must divide n_heads")
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.qkv = nn.Linear(dim, dim * 3)
        self.out = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, allow: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: ``(B, L, D)``。
            allow: ``(L, L)`` 或 ``(B, L, L)``，``True``=允许注意。

        Returns:
            y: ``(B, L, D)``。
        """
        B, L, D = x.shape
        qkv = self.qkv(x).view(B, L, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim**-0.5)
        if allow.ndim == 2:
            mask = allow.view(1, 1, L, L)
        else:
            mask = allow.unsqueeze(1)
        attn = attn.masked_fill(~mask, float("-inf"))
        attn = self.drop(attn.softmax(dim=-1))
        h = (attn @ v).transpose(1, 2).reshape(B, L, D)
        return self.out(h)


class TransfusionBlock(nn.Module):
    """标准 Pre-LN Transformer 块 + 外部注意力掩码。"""

    def __init__(self, dim: int, n_heads: int, dropout: float) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = MaskedAttention(dim, n_heads, dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim),
        )

    def forward(self, x: torch.Tensor, allow: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: ``(B, L, D)``。
            allow: 注意力允许掩码。

        Returns:
            y: ``(B, L, D)``。
        """
        x = x + self.attn(self.norm1(x), allow)
        x = x + self.mlp(self.norm2(x))
        return x


class TinyTransfusion(nn.Module):
    """
    同一套权重上的双目标模型：
    - 文本位：词表 logits → CE（NTP）
    - 图像位：连续速度头 → 流匹配 MSE
    """

    def __init__(self, cfg: TinyTransfusionConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.text_emb = nn.Embedding(cfg.text_vocab_size, cfg.dim)
        self.patch = PatchEmbed(cfg)
        self.time_mlp = nn.Sequential(
            nn.Linear(1, cfg.dim),
            nn.SiLU(),
            nn.Linear(cfg.dim, cfg.dim),
        )
        self.blocks = nn.ModuleList(
            [TransfusionBlock(cfg.dim, cfg.n_heads, cfg.dropout) for _ in range(cfg.n_layers)]
        )
        self.norm = nn.LayerNorm(cfg.dim)
        self.text_head = nn.Linear(cfg.dim, cfg.text_vocab_size, bias=False)
        self.image_head = nn.Linear(cfg.dim, cfg.patch_dim)

    def _time_embed(self, t: torch.Tensor, n_img: int) -> torch.Tensor:
        """
        Args:
            t: ``(B,)`` in ``[0,1]``。
            n_img: 图像 patch 数。

        Returns:
            te: ``(B, n_img, D)`` 加到每个图像 token 上的时间条件。
        """
        te = self.time_mlp(t.view(-1, 1))  # (B, D)
        return te.unsqueeze(1).expand(-1, n_img, -1)

    def pack_embeddings(
        self,
        text_ids: torch.Tensor,
        image_patches: torch.Tensor | None,
        is_image: torch.Tensor,
        t: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """
        按 ``is_image`` 把离散文本嵌入与连续图像嵌入填进同一序列。

        Args:
            text_ids: ``(B, L)``；图像槽位可填任意占位 id（会被忽略）。
            image_patches: ``(B, N, patch_dim)`` 当前（可能加噪的）连续 patch；
                若本序列无图像可为 ``None``。
            is_image: ``(L,)``。
            t: ``(B,)`` 流匹配时间；有图像时必填。

        Returns:
            x: ``(B, L, D)``。
        """
        B, L = text_ids.shape
        x = self.text_emb(text_ids)
        if is_image.any():
            if image_patches is None or t is None:
                raise ValueError("image_patches and t required when sequence has images")
            n_img = int(is_image.sum().item())
            if image_patches.size(1) != n_img:
                raise ValueError("image_patches N must match #image positions")
            img_h = self.patch.embed(image_patches) + self._time_embed(t, n_img)
            # 广播写入：所有 batch 共享同一布局
            x = x.clone()
            x[:, is_image] = img_h
        return x

    def forward(
        self,
        text_ids: torch.Tensor,
        image_patches: torch.Tensor | None,
        is_image: torch.Tensor,
        image_block_id: torch.Tensor,
        t: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        """
        Args:
            text_ids: ``(B, L)``。
            image_patches: ``(B, N, P)`` 或 ``None``。
            is_image: ``(L,)``。
            image_block_id: ``(L,)``。
            t: ``(B,)``；有图像时需要。

        Returns:
            text_logits: ``(B, L, V)``（图像位上的 logits 无监督意义）。
            v_pred: ``(B, N, P)`` 图像速度预测；无图像时为 ``None``。
        """
        allow = build_transfusion_attn_mask(is_image, image_block_id)
        x = self.pack_embeddings(text_ids, image_patches, is_image, t)
        for blk in self.blocks:
            x = blk(x, allow)
        h = self.norm(x)
        text_logits = self.text_head(h)
        v_pred: torch.Tensor | None = None
        if is_image.any():
            v_pred = self.image_head(h[:, is_image])
        return text_logits, v_pred

    def compute_losses(
        self,
        text_ids: torch.Tensor,
        clean_patches: torch.Tensor,
        is_image: torch.Tensor,
        image_block_id: torch.Tensor,
        t: torch.Tensor | None = None,
    ) -> dict[str, torch.Tensor]:
        """
        一次前向同时算 NTP + 流匹配损失。

        Args:
            text_ids: ``(B, L)`` 完整离散序列（含边界符；图像槽为占位）。
            clean_patches: ``(B, N, P)`` 干净连续 patch ``x0``。
            is_image: ``(L,)``。
            image_block_id: ``(L,)``。
            t: ``(B,)``；``None`` 则 ``U(0,1)`` 采样。

        Returns:
            dict，含 ``loss`` / ``text_loss`` / ``image_loss`` / ``t``。
        """
        B = text_ids.size(0)
        if t is None:
            t = torch.rand(B, device=text_ids.device)
        xt, v_tgt, _ = sample_flow_matching(clean_patches, t)
        logits, v_pred = self.forward(text_ids, xt, is_image, image_block_id, t)
        assert v_pred is not None

        # 文本 CE：仅在非图像位做 next-token（移位）
        # 简化：对整段离散 id 做因果 CE，但把「预测落在图像槽」的位置 mask 掉
        shift_logits = logits[:, :-1, :]
        shift_labels = text_ids[:, 1:]
        # 标签位置对应原序列 index+1；若该位是图像则忽略
        label_is_image = is_image[1:]
        valid = ~label_is_image
        text_loss = F.cross_entropy(
            shift_logits[:, valid, :].reshape(-1, shift_logits.size(-1)),
            shift_labels[:, valid].reshape(-1),
        )
        image_loss = F.mse_loss(v_pred, v_tgt)
        loss = (
            self.cfg.text_loss_weight * text_loss
            + self.cfg.image_loss_weight * image_loss
        )
        return {
            "loss": loss,
            "text_loss": text_loss,
            "image_loss": image_loss,
            "t": t,
        }


print("TinyTransfusion ready")


## 4. 序列打包与并行去噪推理（预留图像槽）


In [ ]:
def build_transfusion_batch(
    text_prefix: torch.Tensor,
    clean_images: torch.Tensor,
    text_suffix: torch.Tensor,
    patch_embed: PatchEmbed,
    cfg: TinyTransfusionConfig,
) -> dict[str, torch.Tensor]:
    """
    构造训练 batch：文本 id 序列 + 干净连续 patch + 模态标记。

    布局：``prefix + <image> + N 占位 + </image> + suffix``。
    图像槽上的 id 仅占位，真实内容走 ``clean_patches``。

    Args:
        text_prefix: ``(B, Lp)``。
        clean_images: ``(B, 3, H, W)``。
        text_suffix: ``(B, Ls)``。
        patch_embed: patch 工具。
        cfg: 配置。

    Returns:
        dict：``text_ids`` / ``clean_patches`` / ``is_image`` / ``image_block_id``。
    """
    B = text_prefix.size(0)
    device = text_prefix.device
    start = torch.full((B, 1), cfg.id_img_start, device=device, dtype=torch.long)
    end = torch.full((B, 1), cfg.id_img_end, device=device, dtype=torch.long)
    # 图像槽填 0 占位
    img_pad = torch.zeros(B, cfg.num_patches, device=device, dtype=torch.long)
    text_ids = torch.cat([text_prefix, start, img_pad, end, text_suffix], dim=1)
    clean_patches = patch_embed.encode(clean_images)
    is_image, image_block_id = modality_flags_from_layout(
        text_prefix_len=text_prefix.size(1),
        num_patches=cfg.num_patches,
        text_suffix_len=text_suffix.size(1),
        block_id=0,
        device=device,
    )
    return {
        "text_ids": text_ids,
        "clean_patches": clean_patches,
        "is_image": is_image,
        "image_block_id": image_block_id,
    }


@torch.no_grad()
def denoise_image_parallel(
    model: TinyTransfusion,
    text_prefix: torch.Tensor,
    num_steps: int = 16,
) -> torch.Tensor:
    """
    预留 ``N`` 个图像槽，从 ``t=1`` 噪声并行积分到 ``t=0``。

    Args:
        model: Transfusion 玩具模型。
        text_prefix: ``(B, Lp)`` 条件文本（其后接 ``<image>...``）。
        num_steps: 欧拉步数（教学用小步数）。

    Returns:
        images: ``(B, 3, H, W)`` 由连续 patch 拼回的图。
    """
    cfg = model.cfg
    B = text_prefix.size(0)
    device = text_prefix.device
    empty_suffix = text_prefix.new_zeros(B, 0)
    start = torch.full((B, 1), cfg.id_img_start, device=device, dtype=torch.long)
    end = torch.full((B, 1), cfg.id_img_end, device=device, dtype=torch.long)
    img_pad = torch.zeros(B, cfg.num_patches, device=device, dtype=torch.long)
    text_ids = torch.cat([text_prefix, start, img_pad, end, empty_suffix], dim=1)
    is_image, image_block_id = modality_flags_from_layout(
        text_prefix_len=text_prefix.size(1),
        num_patches=cfg.num_patches,
        text_suffix_len=0,
        block_id=0,
        device=device,
    )

    xt = torch.randn(B, cfg.num_patches, cfg.patch_dim, device=device)
    # t: 1 → 0
    ts = torch.linspace(1.0, 0.0, num_steps + 1, device=device)
    for i in range(num_steps):
        t_cur = ts[i].expand(B)
        t_next = ts[i + 1]
        dt = float((t_next - ts[i]).item())
        _, v_pred = model.forward(text_ids, xt, is_image, image_block_id, t_cur)
        assert v_pred is not None
        xt = flow_step(xt, v_pred, dt)
    return model.patch.decode_patches(xt)


@torch.no_grad()
def greedy_next_text_token(
    model: TinyTransfusion,
    text_ids: torch.Tensor,
    is_image: torch.Tensor,
    image_block_id: torch.Tensor,
) -> torch.Tensor:
    """
    无图像条件下的下一步文本贪心采样（示意 AR 文本头）。

    Args:
        model: 模型。
        text_ids: ``(B, L)`` 当前纯文本（或已生成前缀）。
        is_image: ``(L,)`` 应全为 False。
        image_block_id: ``(L,)``。

    Returns:
        next_ids: ``(B, 1)``。
    """
    logits, _ = model.forward(text_ids, None, is_image, image_block_id, t=None)
    return logits[:, -1, :].argmax(dim=-1, keepdim=True)


print("pack + parallel denoise ready")


## 5. 可选：MMDiT 风格「模态分权重」示意（同注意力、异 MLP）


In [ ]:
class ModalitySplitMLP(nn.Module):
    """同一序列位置按模态走不同 MLP（MMDiT 思想的极简版）。"""

    def __init__(self, dim: int) -> None:
        super().__init__()
        self.mlp_text = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim),
        )
        self.mlp_image = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim),
        )

    def forward(self, x: torch.Tensor, is_image: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: ``(B, L, D)``。
            is_image: ``(L,)``。

        Returns:
            y: ``(B, L, D)``。
        """
        y = x.clone()
        if (~is_image).any():
            y[:, ~is_image] = self.mlp_text(x[:, ~is_image])
        if is_image.any():
            y[:, is_image] = self.mlp_image(x[:, is_image])
        return y


class TinyMMDiTBlock(nn.Module):
    """联合注意力 + 模态分叉 MLP。"""

    def __init__(self, dim: int, n_heads: int, dropout: float) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = MaskedAttention(dim, n_heads, dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = ModalitySplitMLP(dim)

    def forward(
        self,
        x: torch.Tensor,
        allow: torch.Tensor,
        is_image: torch.Tensor,
    ) -> torch.Tensor:
        """
        Args:
            x: ``(B, L, D)``。
            allow: 注意力掩码。
            is_image: ``(L,)``。

        Returns:
            y: ``(B, L, D)``。
        """
        x = x + self.attn(self.norm1(x), allow)
        x = x + self.mlp(self.norm2(x), is_image)
        return x


print("MMDiT-style split MLP ready")


## 6. 冒烟测试


In [ ]:
def smoke_test() -> None:
    """验证 Transfusion 教学要点：掩码、双损失、并行去噪。"""
    torch.manual_seed(0)
    cfg = TinyTransfusionConfig()
    model = TinyTransfusion(cfg)
    model.eval()

    print("=== attention mask (prefix=2, N=4 toy layout) ===")
    # 临时用更小 N 只测掩码形状逻辑：手写 flags
    is_image = torch.tensor([False, False, False, True, True, False])
    # positions: t0 t1 <img> p0 p1 </img>
    image_block_id = torch.tensor([-1, -1, -1, 0, 0, -1])
    allow = build_transfusion_attn_mask(is_image, image_block_id)
    # 图像内双向：p0 可见 p1
    assert bool(allow[3, 4].item()) and bool(allow[4, 3].item())
    # 文本因果：t1 不可见未来文本位 2？ 位2是 <img> 文本且 j=2>i=1 → 不可见
    assert not bool(allow[1, 2].item())
    # 图像可见前缀文本
    assert bool(allow[3, 0].item())
    # 前缀不可见未来图像？ t0 看 p0：j=3>i=0 → False
    assert not bool(allow[0, 3].item())
    print(f"allow={tuple(allow.shape)} OK")

    print("\n=== dual loss forward ===")
    model.train()
    prefix = torch.randint(5, cfg.text_vocab_size, (2, 3))
    suffix = torch.randint(5, cfg.text_vocab_size, (2, 2))
    images = torch.randn(2, 3, cfg.image_size, cfg.image_size)
    batch = build_transfusion_batch(prefix, images, suffix, model.patch, cfg)
    out = model.compute_losses(
        batch["text_ids"],
        batch["clean_patches"],
        batch["is_image"],
        batch["image_block_id"],
    )
    print(
        f"text_ids={tuple(batch['text_ids'].shape)} "
        f"patches={tuple(batch['clean_patches'].shape)}"
    )
    print(
        f"loss={out['loss'].item():.4f} "
        f"text={out['text_loss'].item():.4f} "
        f"image={out['image_loss'].item():.4f}"
    )
    assert out["loss"].ndim == 0
    out["loss"].backward()
    grad_ok = any(p.grad is not None and p.grad.abs().sum() > 0 for p in model.parameters())
    assert grad_ok
    print("backward OK")

    print("\n=== parallel denoise (preallocated slots) ===")
    model.eval()
    prompt = torch.randint(5, cfg.text_vocab_size, (1, 4))
    rendered = denoise_image_parallel(model, prompt, num_steps=8)
    print(f"rendered={tuple(rendered.shape)}")
    assert rendered.shape == (1, 3, cfg.image_size, cfg.image_size)

    print("\n=== modality-split MLP ===")
    blk = TinyMMDiTBlock(cfg.dim, cfg.n_heads, 0.0)
    x = torch.randn(1, batch["text_ids"].size(1), cfg.dim)
    allow_full = build_transfusion_attn_mask(batch["is_image"], batch["image_block_id"])
    y = blk(x, allow_full, batch["is_image"])
    print(f"mmdit_block out={tuple(y.shape)}")
    assert y.shape == x.shape

    print("SMOKE TEST OK")


smoke_test()
